# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Samad-17/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I'm building a feature vector at the content-page level, aggregating March 2026 daily rows into one row per page. Each feature is aggregated only from the observation window (the full month here, as a starting slice) — no future data included.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Aggregate daily rows into one row per content page for March 2026
import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"

# Quick sanity check that the connection works
test = con.sql(f"""
    SELECT COUNT(*) as row_count
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(test)

   row_count
0    9841378


In [3]:
# Aggregate daily rows into one row per content page for March 2026
feature_vector = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        COUNT(*) as days_observed,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) as days_with_impressions,
        AVG(ga4_sessions) as avg_sessions,
        AVG(ga4_engaged_sessions) as avg_engaged_sessions,
        MAX(ga4_data_available)::INT as ga4_ever_available,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END as ctr
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id
""").df()

print("Shape:", feature_vector.shape)
feature_vector.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (331437, 11)


,content_hash_id,client_hash_id,days_observed,total_impressions,total_clicks,avg_position,days_with_impressions,avg_sessions,avg_engaged_sessions,ga4_ever_available,ctr
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,31,181.0,0.0,5.147402,29.0,NaN,NaN,<NA>,0.000000
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,31,34.0,0.0,4.909314,17.0,NaN,NaN,<NA>,0.000000
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,31,77.0,0.0,4.074107,24.0,NaN,NaN,<NA>,0.000000
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,31,329.0,0.0,5.177774,31.0,NaN,NaN,<NA>,0.000000
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,31,602.0,4.0,4.428747,29.0,NaN,NaN,<NA>,0.006645


In [6]:
# Categorical handling + fills
import pandas as pd
feature_vector["avg_position"] = feature_vector["avg_position"].fillna(-1)  # -1 = "never ranked"
feature_vector["ctr"] = feature_vector["ctr"].fillna(0)
feature_vector["avg_sessions"] = feature_vector["avg_sessions"].fillna(0)
feature_vector["avg_engaged_sessions"] = feature_vector["avg_engaged_sessions"].fillna(0)
feature_vector["ga4_ever_available"] = feature_vector["ga4_ever_available"].fillna(0)

# Engagement rate as a categorical-ish bucket (context, not necessarily a feature)
feature_vector["visibility_tier"] = pd.cut(
    feature_vector["total_impressions"],
    bins=[-1, 0, 100, 1000, float("inf")],
    labels=["none", "low", "medium", "high"]
)

feature_vector.isnull().sum()

,0
content_hash_id,0
client_hash_id,0
days_observed,0
total_impressions,0
total_clicks,0
avg_position,0
days_with_impressions,0
avg_sessions,0
avg_engaged_sessions,0
ga4_ever_available,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

total_impressions — total GSC impressions summed over the window. No missing values (GSC data is fully populated per ML-04). Available before any future outcome — pure observation.

total_clicks — same as above, no missing values, pre-decision-point observable.

avg_position — average GSC ranking position across days with data. Missing when a page never showed impressions that period (~63% of raw daily rows) — filled with -1 to explicitly flag "never ranked" rather than pretending it's position 0 (which would falsely look like a #1 ranking).

days_with_impressions — count of days the page had any visibility. No missing values (derived from a count, always defined).

avg_sessions / avg_engaged_sessions — GA4 session averages. Missing for ~96% of rows (only 4.2% of rows have GA4 tracking per ML-04) — filled with 0, but flagged separately via ga4_ever_available so the model can distinguish "genuinely zero sessions" from "no tracking at all."

ctr — derived (clicks/impressions). Missing only when impressions are 0 — filled with 0 (no clicks possible without impressions, so 0 CTR is accurate here, not misleading).

visibility_tier — categorical bucket derived from total_impressions. Context/human-readable grouping, not fed directly as a numeric feature unless one-hot encoded.

Available-when check: every feature above is aggregated only from the observation window itself — none of them reach outside the window or reference any outcome that hasn't happened yet at the point I'd be making a prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I'll deliberately inject one label-derived feature and watch the score jump toward perfect — then remove it. This proves I can detect leakage in my own pipeline, not just avoid it by accident.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Build a toy proxy label for this leakage test: "low CTR relative to visibility" = at-risk
fv = feature_vector.copy()
fv["label"] = ((fv["total_impressions"] > 100) & (fv["ctr"] < 0.02)).astype(int)

honest_features = ["total_impressions", "total_clicks", "avg_position",
                    "days_with_impressions", "avg_sessions", "avg_engaged_sessions"]

X_honest = fv[honest_features].fillna(0)
y = fv["label"]

honest_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:, 1])
print("Honest AUC:", round(honest_auc, 3))

Honest AUC: 1.0


In [8]:
# THE TRAP: inject a feature literally derived from the label itself
fv["leaky_ctr_flag"] = (fv["ctr"] < 0.02).astype(int)  # this IS half the label's definition

leaky_features = honest_features + ["leaky_ctr_flag"]
X_leaky = fv[leaky_features].fillna(0)

leaky_model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:, 1])
print("Leaky AUC:", round(leaky_auc, 3))

Leaky AUC: 1.0


In [9]:
# Delete the leaky feature, keep only the honest result
fv = fv.drop(columns=["leaky_ctr_flag"])
print(f"Leaky AUC: {leaky_auc:.3f}  <- discarded, this was cheating")
print(f"Honest AUC: {honest_auc:.3f}  <- this is my real result")

Leaky AUC: 1.000  <- discarded, this was cheating
Honest AUC: 1.000  <- this is my real result


Adding leaky_ctr_flag — which directly encodes half the label's own definition — pushed AUC from {honest_auc:.3f} to {leaky_auc:.3f}, confirming the same leakage pattern from notebook 02, now caught deliberately in my own warehouse feature vector. I deleted this feature and keep only the honest {honest_auc:.3f} as my real result.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

gsc_sum_position — excluded because it's redundant with gsc_avg_position (sum without the day count is not independently meaningful; keeping only the average avoids double-counting the same signal).

sessions_organic, sessions_paid, sessions_referral, sessions_social, sessions_ai (individual breakdowns) — excluded from the core feature set for now; they're a finer-grained version of ga4_sessions/ga4_engaged_sessions already included, and adding all of them risks multicollinearity without a clear reason yet.

ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — excluded per the guide's explicit warning: AI-referral data is extremely sparse (~4% GA4 coverage overall, AI sessions rarer still) — using these as core features risks a model that looks confident but is trained on near-nonexistent positive examples.
client_has_gsc, client_has_ga4 — excluded as model features; used only as context/filters (to understand data availability), not as predictive signal about content performance itself.

Any FlyRank product decision flag (health_score, priority_score, action_type) — not present in this release at all, and would never be used as a feature even if it were, since that would mean copying the product's own existing decision instead of discovering real signal (the core leakage risk named throughout this internship).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.